# 🔢 Notebook 3: Feature Extraction con TF-IDF

## 🎯 Objetivos de este Notebook

En este notebook aprenderás:
1. ✅ **¿Por qué** convertir texto a números?
2. ✅ **Bag of Words** (Bolsa de Palabras)
3. ✅ **TF-IDF**: Qué es y cómo funciona
4. ✅ Crear vectores TF-IDF con ejemplos reales
5. ✅ Visualizar features y palabras importantes

⏱️ **Tiempo estimado**: 25 minutos

---

## 💡 El Problema: Las Computadoras Solo Entienden Números

### ❌ Esto NO funciona:

```python
modelo.fit(["I love this movie", "Terrible film"], [1, 0])
# Error: El modelo solo acepta números, no texto
```

### ✅ Necesitamos convertir:

```python
"I love this movie" → [0.5, 0.8, 0.3, 0.0, 0.0, ...]  # Vector de números
modelo.fit(vectores_numericos, labels)  # ¡Ahora sí funciona!
```

---

## 🔧 Setup Inicial

In [ ]:
# Importar librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
from pathlib import Path

# ============================================================================
# CONFIGURACIÓN DE RUTAS - Solución robusta para Jupyter
# ============================================================================

def find_project_root():
    """Encuentra la raíz del proyecto buscando config.py y src/"""
    current = Path.cwd()
    
    # Buscar hacia arriba hasta 5 niveles
    for _ in range(5):
        config_exists = (current / 'config.py').exists()
        src_exists = (current / 'src').exists()
        
        if config_exists and src_exists:
            return current
        
        # También verificar si estamos dentro de sentiment-analysis
        if current.name == 'sentiment-analysis' and src_exists:
            return current
            
        current = current.parent
    
    # Si no encuentra, asumir que es el directorio actual
    return Path.cwd()

# Encontrar y configurar la ruta del proyecto
project_root = find_project_root()
print(f"📁 Proyecto encontrado en: {project_root}")

# Agregar al path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Cambiar el working directory
os.chdir(str(project_root))

# Importar módulos
from src import feature_extraction, text_preprocessing
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

print("✅ Setup completo!")

## 📊 Método 1: Bag of Words (Bolsa de Palabras)

La idea más simple: **Contar cuántas veces aparece cada palabra**.

### Ejemplo con 3 reviews:

In [ ]:
# Reviews de ejemplo
reviews = [
    "I love this movie",      # Review 1 (positiva)
    "I hate this movie",      # Review 2 (negativa)
    "Great movie I love it"   # Review 3 (positiva)
]

# Crear Bag of Words
vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(reviews)

# Ver vocabulario (palabras únicas)
vocab = vectorizer.get_feature_names_out()
print("📚 Vocabulario (palabras únicas):")
print(f"   {list(vocab)}")
print(f"\n📊 Total de palabras únicas: {len(vocab)}")

### Convertir a DataFrame para ver mejor:

In [ ]:
# Crear tabla
df_bow = pd.DataFrame(
    bow_matrix.toarray(),
    columns=vocab,
    index=[f"Review {i+1}" for i in range(len(reviews))]
)

print("\n📊 MATRIZ BAG OF WORDS:")
print("="*70)
print(df_bow)
print("="*70)

print("\n💡 Explicación:")
print("   • Cada columna = una palabra del vocabulario")
print("   • Cada fila = una review")
print("   • Números = cuántas veces aparece la palabra")
print("\n   Ejemplo: 'love' aparece 1 vez en Review 1, 0 en Review 2, 1 en Review 3")

### Visualizar la matriz:

In [ ]:
# Heatmap de Bag of Words
plt.figure(figsize=(12, 4))
sns.heatmap(df_bow, annot=True, fmt='d', cmap='YlOrRd', cbar_kws={'label': 'Frecuencia'})
plt.title('Bag of Words - Frecuencia de Palabras', fontsize=14, fontweight='bold')
plt.xlabel('Palabras', fontsize=12)
plt.ylabel('Reviews', fontsize=12)
plt.tight_layout()
plt.show()

### ⚠️ Problema con Bag of Words:

Todas las palabras tienen el **mismo peso**:

```
"movie" aparece en las 3 reviews → Peso: 1
"great" aparece en 1 review     → Peso: 1
```

Pero **"great" es más discriminativa** que "movie"!

### Solución: **TF-IDF**

---

## 📈 Método 2: TF-IDF (Term Frequency - Inverse Document Frequency)

### 🧮 Fórmula:

```
TF-IDF(palabra, documento) = TF(palabra, documento) × IDF(palabra)
```

Donde:
- **TF** = Frecuencia de la palabra en ESTE documento
- **IDF** = Qué tan rara es la palabra en TODOS los documentos

### 💡 Intuición:

| Palabra | Aparece en... | IDF | Importancia |
|---------|---------------|-----|-------------|
| "movie" | Todas las reviews | Bajo (0.1) | Poco discriminativa |
| "masterpiece" | 1 review | Alto (2.5) | ¡Muy discriminativa! |

---

## 🎯 TF-IDF con el Mismo Ejemplo:

In [ ]:
# Crear TF-IDF
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(reviews)

# Crear tabla
df_tfidf = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f"Review {i+1}" for i in range(len(reviews))]
)

print("\n📈 MATRIZ TF-IDF:")
print("="*70)
print(df_tfidf.round(3))  # Redondear a 3 decimales
print("="*70)

print("\n💡 Diferencias con Bag of Words:")
print("   • Los valores NO son enteros (son pesos)")
print("   • Palabras raras tienen valores más altos")
print("   • Palabras comunes tienen valores más bajos")

## 📊 Comparación Visual: BoW vs TF-IDF

In [ ]:
# Comparar para la primera review
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bag of Words
axes[0].bar(df_bow.columns, df_bow.iloc[0], color='steelblue', alpha=0.7)
axes[0].set_title('Bag of Words - Review 1', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Palabras')
axes[0].set_ylabel('Frecuencia')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# TF-IDF
axes[1].bar(df_tfidf.columns, df_tfidf.iloc[0], color='coral', alpha=0.7)
axes[1].set_title('TF-IDF - Review 1', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Palabras')
axes[1].set_ylabel('Peso TF-IDF')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Observación:")
print("   • BoW: Todas las palabras presentes tienen valor 1")
print("   • TF-IDF: Palabras tienen pesos diferentes según su importancia")

## 🔬 Ejemplo con Reviews Reales

Ahora probemos con reviews más realistas:

In [ ]:
# Reviews más realistas
reviews_reales = [
    "This movie is excellent and the acting is superb",
    "Terrible waste of time awful acting",
    "Great film loved the story",
    "Boring movie not recommended",
    "Masterpiece brilliant performance amazing"
]

labels = [1, 0, 1, 0, 1]  # 1=positivo, 0=negativo

# Preprocesar
reviews_procesadas = [
    text_preprocessing.preprocess_text(r) for r in reviews_reales
]

print("📝 Reviews preprocesadas:")
for i, (orig, proc) in enumerate(zip(reviews_reales, reviews_procesadas)):
    sentiment = "Positivo ✅" if labels[i] == 1 else "Negativo ❌"
    print(f"\n{i+1}. {sentiment}")
    print(f"   Original:  '{orig}'")
    print(f"   Procesada: '{proc}'")

## 🎯 Crear TF-IDF para Reviews Reales:

In [ ]:
# Crear vectorizador TF-IDF
tfidf_vec = TfidfVectorizer(max_features=20)  # Top 20 palabras
X_tfidf = tfidf_vec.fit_transform(reviews_procesadas)

# Convertir a DataFrame
df_tfidf_real = pd.DataFrame(
    X_tfidf.toarray(),
    columns=tfidf_vec.get_feature_names_out(),
    index=[f"Review {i+1} ({'Pos' if labels[i]==1 else 'Neg'})" for i in range(len(reviews_reales))]
)

print("\n📊 MATRIZ TF-IDF (Reviews Reales):")
print("="*100)
print(df_tfidf_real.round(3))
print("="*100)

## 🔍 Análisis: Palabras Más Importantes

In [ ]:
# Calcular importancia promedio de cada palabra
importancia_palabras = df_tfidf_real.mean().sort_values(ascending=False)

print("\n🏆 TOP 10 PALABRAS MÁS IMPORTANTES (promedio TF-IDF):")
print("="*70)
for palabra, score in importancia_palabras.head(10).items():
    print(f"   • '{palabra}': {score:.4f}")

# Visualizar
plt.figure(figsize=(12, 5))
importancia_palabras.head(10).plot(kind='barh', color='teal', alpha=0.7)
plt.xlabel('Importancia Promedio (TF-IDF)', fontsize=12)
plt.title('Top 10 Palabras Más Importantes', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 🎨 Heatmap de Features TF-IDF

In [ ]:
# Heatmap
plt.figure(figsize=(14, 6))
sns.heatmap(df_tfidf_real, annot=True, fmt='.2f', cmap='RdYlGn', 
            cbar_kws={'label': 'TF-IDF Score'})
plt.title('TF-IDF Features - Reviews Reales', fontsize=14, fontweight='bold')
plt.xlabel('Palabras', fontsize=12)
plt.ylabel('Reviews', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\n💡 En el heatmap:")
print("   • Verde = TF-IDF alto (palabra importante en esa review)")
print("   • Amarillo = TF-IDF medio")
print("   • Rojo = TF-IDF bajo o cero (palabra no aparece)")

## 🧪 Experimento: ¿Cómo Afecta el Vocabulario?

Probemos con diferentes tamaños de vocabulario:

In [ ]:
# Probar con diferentes tamaños de vocabulario
vocab_sizes = [5, 10, 15, 20, None]  # None = todas las palabras

results = []
for size in vocab_sizes:
    vec = TfidfVectorizer(max_features=size)
    X = vec.fit_transform(reviews_procesadas)
    
    actual_size = X.shape[1]
    sparsity = (1 - X.nnz / (X.shape[0] * X.shape[1])) * 100
    
    results.append({
        'Tamaño solicitado': size if size else 'Todas',
        'Features creadas': actual_size,
        'Sparsity (%)': f"{sparsity:.1f}%"
    })

df_vocab = pd.DataFrame(results)
print("\n📊 IMPACTO DEL TAMAÑO DEL VOCABULARIO:")
print("="*70)
print(df_vocab.to_string(index=False))
print("="*70)

print("\n💡 Sparsity = % de ceros en la matriz")
print("   • Vocabulario pequeño → Menos features, menos sparse")
print("   • Vocabulario grande → Más features, más sparse")

## 🎯 Ejercicio Interactivo

**Crea TF-IDF para tus propias reviews:**

In [ ]:
# 👇 ESCRIBE TUS PROPIAS REVIEWS:
mis_reviews = [
    "Amazing movie with great acting",
    "Terrible film very disappointing",
    "Masterpiece absolutely brilliant"
]

# Preprocesar
mis_reviews_proc = [text_preprocessing.preprocess_text(r) for r in mis_reviews]

# Crear TF-IDF
vec = TfidfVectorizer()
X = vec.fit_transform(mis_reviews_proc)

# Ver resultados
df_resultado = pd.DataFrame(
    X.toarray(),
    columns=vec.get_feature_names_out(),
    index=[f"Review {i+1}" for i in range(len(mis_reviews))]
)

print("\n📊 TF-IDF de tus reviews:")
print("="*70)
print(df_resultado.round(3))
print("="*70)

## 📊 Resumen de lo Aprendido

En este notebook aprendiste:

✅ **Por qué Feature Extraction**:
- Modelos de ML solo entienden números
- Necesitamos convertir texto → vectores numéricos

✅ **Bag of Words**:
- Cuenta frecuencia de palabras
- Simple pero limitado (todas las palabras pesan igual)

✅ **TF-IDF**:
- TF = Frecuencia en documento
- IDF = Importancia global
- Palabras raras → peso alto
- Palabras comunes → peso bajo

✅ **Implementación Práctica**:
- `TfidfVectorizer` de scikit-learn
- `max_features` controla tamaño del vocabulario
- Matriz sparse (mayoría ceros)

✅ **Visualizaciones**:
- Heatmaps para ver features
- Comparación BoW vs TF-IDF
- Palabras más importantes

---

## 🎓 Próximo Paso

En el **Notebook 4** aprenderás:
- 🤖 Entrenar modelos clásicos (Naive Bayes, SVM)
- 📊 Usar features TF-IDF para clasificación
- 🎯 Evaluar modelos (accuracy, precision, recall)
- 🔍 Ver qué palabras son más importantes para cada clase

**¡Nos vemos en el siguiente notebook!** 🚀